# Phase 3 — Exploratory Data Analysis (EDA)
## Patterns of Recorded Civilian Targeting in Gaza

**DS312 · Group 7 · Phase 3 notebook template**

This notebook continues from Phase 2. It first loads and checks the saved Phase 2 CSV outputs, then uses each table for the questions it can answer. The analysis moves from event-level descriptions to group comparisons, weekly patterns, and spatial exploration. Replace prompts with results from the saved project snapshot before submission.

> **Reading path:** project questions → Phase 2 CSVs and their roles → data checks → univariate → bivariate → grouped, temporal, and spatial analysis → hypothesis tests → key insights and Phase 4 plan.

## 3.1 Purpose, audience, and decision

### Purpose
Describe patterns, relationships, and unusual periods in the cleaned Gaza data. Use the evidence to refine the Phase 4 plan for spatial clustering and weekly anomaly detection.

### Audience and decision
The audience is human-rights documentation teams, civilian-protection analysts, investigative journalists, and research coordinators. The findings should help prioritize locations, periods, and incident groups for source verification and contextual investigation.

### Interpretation boundary
This notebook describes **recorded ACLED events**. The project focal-event flag is a reproducible coding rule, not proof of intent, responsibility, or legal classification.

### Narrative sequence
1. Read the Phase 2 outputs and confirm their units, dates, and intended use.
2. Describe individual variables before comparing groups.
3. Compare periods, event types, actors, and severity with explicit denominators.
4. Examine district-week and spatial patterns using the tables prepared for those units.
5. Test two exploratory hypotheses and use the results to refine Phase 4.

## 3.2 Phase 3 requirements and definition of done

This notebook is complete when it:

1. Uses the Phase 2 processed outputs and explains which table answers each question.
2. Includes univariate, bivariate, multivariate/grouped, temporal, and spatial exploration relevant to the business questions.
3. Presents **at least 8 purposeful visualizations**. Every figure has a question-led title, labeled axes/legend, and a 1–2 sentence interpretation.
4. States and tests **at least 2 hypotheses**, reporting test, assumptions/checks, p-value, effect size where appropriate, and a plain-language conclusion.
5. Includes **5–8 key insights** and a Phase 4 modeling plan grounded in the evidence.
6. Documents limitations and avoids causal language for observational associations.
7. Runs top-to-bottom reproducibly; paths, date parsing, and any filtering are explicit.

### Definition of done checklist
- [ ] Each chart answers a stated question and has an interpretation.
- [ ] Denominators match the event group, area, and period shown.
- [ ] Sparse categories and small expected test counts are handled transparently.
- [ ] Before/after period lengths and incomplete final week are acknowledged.
- [ ] EDA is exploratory; no future information is used to claim a predictive result.
- [ ] Findings link back to a Phase 1 business question.

## 3.3 Business questions guiding the analysis

Phase 1 established five questions. Phase 2 narrowed the working data to Gaza, so this notebook applies the questions within that scope. Before-and-after comparisons use the cleaned event master; post–October 7 temporal analysis uses the weekly CSVs; spatial exploration uses the land focal-incident CSV.

| ID | Question for Phase 3 | Main Phase 2 output | Phase 4 use |
|---|---|---|---|
| BQ1 | How do recorded focal-event counts and their share of political-violence events differ before and after October 7, 2023 in Gaza? | Cleaned event master | Descriptive baseline and sensitivity questions |
| BQ2 | Which recorded actors and event/sub-event types recur among focal events? | Cleaned event master | Interpretable event profiles; review notes before assigning roles |
| BQ3 | Where are post–October 7 land focal events concentrated, and how does precision affect the view? | Focal-incident table | DBSCAN settings and precision sensitivity |
| BQ4 | Which district-weeks show high or sustained focal activity relative to recent local history? | District-week and Gaza-week tables | Weekly anomaly features and baseline choices |
| BQ5 | Which patterns change under alternative periods, geography, or precision restrictions? | Relevant output table(s) | Robustness checks |

A planned question is not a result. Write each result only after the relevant table and plot have been reviewed.

## 3.4 Read the Phase 2 outputs before exploring them

Phase 2 saved four CSVs in `data/processed/`. They represent different populations and units. Load them in separate cells so it is clear which file supports each part of the analysis. Run this notebook from the project folder so the relative paths resolve.

| Phase 2 CSV | What one row represents | Use it for | Important scope note |
|---|---|---|---|
| `acled_events_clean.csv` | One Gaza event record, across both periods | Event-level variables, before/after comparisons, actor and event-type profiles, fatalities | Retains Gaza events, including coast-coded records and four records without a district. |
| `acled_gaza_week.csv` | One Gaza-wide calendar week after October 7 | Gaza-wide counts, shares, and temporal context | Zero-event weeks are present; final week may be incomplete; share is undefined with zero political events. |
| `acled_district_week.csv` | One known district and calendar week after October 7 | District comparisons, weekly persistence, heatmaps, lag features | District table excludes four records without `admin2`; use it only for district-level analysis. |
| `acled_focal_incidents.csv` | One post–October 7 land focal incident | Coordinate distribution and precision sensitivity | Coast-coded focal events are excluded for land spatial analysis; repeated coordinates are retained. |

The next cells load each file separately. Read the accompanying note before each cell; it explains why that output exists and what it can support.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")

# Run from the project root (the folder containing data/processed/).
DATA_DIR = Path("data/processed")
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_DIR.resolve()}. Run this notebook from the project root."
    )
print("Reading Phase 2 CSVs from:", DATA_DIR.resolve())

### 3.4.1 Cleaned event master — `acled_events_clean.csv`

**Why this file:** it is the event-level Gaza master across the full download window. Use it for variable distributions, event/actor profiles, and the October 7 descriptive comparison. It is not the spatially restricted table.

**Read/check:** parse the saved event dates and restore precision codes as ordered categories; confirm the date span and inspect a few rows.

In [ ]:
master = pd.read_csv(DATA_DIR / "acled_events_clean.csv", parse_dates=["event_date"])
for col in ("geo_precision", "time_precision"):
    master[col] = pd.Categorical(master[col], categories=[1, 2, 3], ordered=True)

print("Clean event master:", master.shape)
print("Event dates:", master.event_date.min().date(), "to", master.event_date.max().date())
display(master[["event_date", "period", "event_type", "fatalities", "focal_event", "admin2"]].head())

### 3.4.2 Gaza-wide weekly table — `acled_gaza_week.csv`

**Why this file:** Phase 2 aligned Gaza-wide event totals to a weekly calendar, including zero-record weeks. Use it to describe weekly counts and shares after October 7 and to inspect the available historical features.

**Read/check:** parse both week boundaries and review completeness, activity, and lag columns.

In [ ]:
gaza_weekly = pd.read_csv(
    DATA_DIR / "acled_gaza_week.csv",
    parse_dates=["week_start", "week_end"],
)
print("Gaza-week table:", gaza_weekly.shape)
display(gaza_weekly[["week_start", "week_end", "complete_week", "political_events",
                     "focal_events", "focal_share", "previous_week_focal",
                     "previous_four_weeks_focal"]].head())

### 3.4.3 District-week table — `acled_district_week.csv`

**Why this file:** each row is a district-week built from records with a known district. Use it to compare district activity over the same calendar weeks and to explore eligible history. Do not substitute it for the Gaza-wide table.

**Read/check:** parse week boundaries and verify the district and week fields before building grouped views.

In [ ]:
district_weekly = pd.read_csv(
    DATA_DIR / "acled_district_week.csv",
    parse_dates=["week_start", "week_end"],
)
print("District-week table:", district_weekly.shape)
print("Districts:", district_weekly.admin2.nunique(),
      "| weeks:", district_weekly.week_start.nunique())
display(district_weekly[["admin2", "week_start", "complete_week", "political_events",
                         "focal_events", "focal_share", "history_eligible"]].head())

### 3.4.4 Land focal-incident table — `acled_focal_incidents.csv`

**Why this file:** Phase 2 prepared post–October 7 focal events for land-based spatial analysis and added coordinates in radians for later haversine clustering. Use the degree coordinates for maps and the precision fields for sensitivity views.

**Read/check:** parse event dates, restore precision categories, then inspect coordinate ranges and repeated locations.

In [ ]:
incidents = pd.read_csv(
    DATA_DIR / "acled_focal_incidents.csv",
    parse_dates=["event_date"],
)
for col in ("geo_precision", "time_precision"):
    incidents[col] = pd.Categorical(incidents[col], categories=[1, 2, 3], ordered=True)
print("Land focal incidents:", incidents.shape)
print("Distinct coordinate pairs:", len(incidents[["latitude", "longitude"]].drop_duplicates()))
print("Precision 1 records:", incidents.geo_precision_1.sum())
display(incidents[["event_date", "location", "latitude", "longitude",
                   "geo_precision", "geo_precision_1"]].head())

### 3.4.5 Loading summary and transition to EDA

The files are now loaded as `master`, `gaza_weekly`, `district_weekly`, and `incidents`. The following audit checks expected fields and key scope facts before charts are made. After that, each EDA section names the table it uses so readers can trace every result back to a Phase 2 output.

## 3.5 Data contract: variables, units, and roles

Use this as the minimum variable contract. Add or remove variables only with a written reason tied to a business question.

| Variable(s) | Type / role in EDA | Use | Handling note |
|---|---|---|---|
| `event_date`, `period`, `month_start`, `week_start` | Date / grouping | Before-after, monthly and weekly trends | Periods differ in duration; compare rates/shares as well as counts. |
| `focal_event` | Boolean outcome for description | Numerator in focal counts and shares | Project-derived rule from targeting flag and exact civilian actor label. |
| `political_violence` | Boolean denominator group | Context and denominator for focal share | Exact token matching was used in Phase 2. |
| `targeting_flagged`, `civilian_targeting` | Boolean / source category | Describe ACLED targeting coding | Blank source value is preserved; derived boolean means recorded flag present. |
| `actor1`, `actor2`, `event_type`, `sub_event_type`, `disorder_type` | Categorical | Actor/event profiles and cross-tabs | Actor labels are high-cardinality; show top groups plus an “Other” display bucket only if documented. |
| `fatalities`, `fatalities_log1p`, `high_fatalities_review` | Count / derived review flag | Severity distribution and comparisons | Retain raw counts; review flags do not mean invalid records. |
| `admin2`, `location` | Geographic categories | District summaries and location descriptions | Four events have missing district; exclude only when district is required. |
| `latitude`, `longitude`, `geo_precision`, `geo_precision_1` | Coordinates / quality | Spatial distribution and sensitivity | Preserve geographic geometry; precision codes are categories, not distances. |
| `time_precision`, `time_precision_1` | Quality category / flag | Sensitivity checks | Do not interpret code values as continuous measurements. |
| `recorded_events`, `political_events`, `focal_events`, `focal_share` | Weekly aggregates | Trends and district-week comparisons | `focal_share` is undefined when denominator is zero. |
| `previous_week_focal`, `previous_four_weeks_focal`, `history_eligible` | Historical features | Explore candidate Phase 4 predictors/baselines | Lagged by Phase 2; use only prior weeks for an alerting feature. |

### Basic integrity review


In [ ]:
# Check the loaded Phase 2 tables before plotting or comparing groups.
required = {
    "master": {"event_date", "period", "event_type", "actor1", "fatalities", "focal_event", "political_violence", "admin2"},
    "gaza_weekly": {"week_start", "week_end", "political_events", "focal_events", "focal_share", "complete_week"},
    "district_weekly": {"admin2", "week_start", "focal_events", "political_events", "focal_share", "history_eligible"},
    "incidents": {"event_date", "location", "latitude", "longitude", "geo_precision", "geo_precision_1"},
}
frames = {"master": master, "gaza_weekly": gaza_weekly,
          "district_weekly": district_weekly, "incidents": incidents}
for name, cols in required.items():
    missing_cols = cols - set(frames[name].columns)
    print(name, "missing expected columns:", sorted(missing_cols))

print("\nEvent-level coverage and missing districts")
print("Missing admin2:", master.admin2.isna().sum())
print("\nFocal-event records by period")
display(master.groupby("period", dropna=False).focal_event.agg(["size", "sum", "mean"]))
print("\nTable shapes:")
for name, frame in frames.items():
    print(f"{name}: {frame.shape}")

## 3.6 Analysis plan and visualization backlog

Complete the “Question / result / interpretation” prompts under each figure. The 10 candidate visuals below give options; select at least 8 and retain only those that reveal something relevant.

| ID | Level | Question and variables | Suggested visual | Requirement / caution |
|---|---|---|---|---|
| V1 | Univariate | What is the distribution of reported fatalities? `fatalities` | Histogram or ECDF; optionally side-by-side with `fatalities_log1p` | Show zeros and long tail; do not delete flagged high counts. |
| V2 | Univariate | Which event types dominate? `event_type` | Ordered count bar chart | Include counts and shares. |
| V3 | Univariate | Which actors recur in focal events? `actor1` / `actor2` | Top-actor bar chart | Explain missing actor2 as structural where applicable; actor labels are coding, not verified roles. |
| V4 | Bivariate | Does focal-event share differ by period? `period`, `focal_event`, `political_violence` | Period rate plot with numerator and denominator | Use political-violence records as denominator; show raw counts too. |
| V5 | Bivariate | Which event types have more focal events? `event_type`, `focal_event` | Grouped/stacked bars or rate chart | Counts can reflect different event-type volumes; include within-type percentages. |
| V6 | Bivariate | How does severity vary by focal status/type? `fatalities`, `focal_event`, `event_type` | Box/violin or ECDF | Heavy skew; show median/IQR or log view and sample sizes. |
| V7 | Multivariate | Which districts have persistent high weeks? `admin2`, `week_start`, `focal_events` | District-week heatmap | Mark zero weeks and report missing-district scope. |
| V8 | Temporal | How do Gaza-wide weekly counts and shares change? `focal_events`, `political_events`, `focal_share` | Line plots with clear denominator/units | Mark incomplete last week; share undefined at zero denominator. |
| V9 | Spatial | Where are land focal incidents recorded? `latitude`, `longitude` | Map/scatter density view | Discuss repeated coordinates and location precision; no inferred borders. |
| V10 | Sensitivity | How does the spatial pattern change for precision 1? `geo_precision_1`, coordinates | Side-by-side map or density plot | Same extent and symbols; describe coverage loss as well as pattern. |

For each visualization include a short Markdown interpretation: **what is visible, how strong/stable it appears, and why it matters for the business question**.

## 3.7 Reusable plotting helpers (optional)

The course guide does not require generic univariate, bivariate, or multivariate EDA functions. It requires purposeful analysis, at least eight interpreted visualizations, two tested hypotheses, and a closing modeling plan. Its reusable-pipeline example appears in the advanced Phase 2 cleaning criteria.

A few small plotting helpers can keep styles consistent. Keep filters, denominators, test choices, and interpretation visible in the analysis cells; each chart still needs its own written interpretation.

In [ ]:
def plot_count(data, column, *, title, order=None, ax=None, normalize=False):
    """Count plot for one categorical variable; normalize=True shows within-column shares."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    counts = data[column].value_counts(dropna=False, normalize=normalize)
    if order is not None:
        counts = counts.reindex(order).dropna()
    counts.sort_values().plot(kind="barh", ax=ax, color="#315A7D")
    ax.set_title(title)
    ax.set_xlabel("Share of records" if normalize else "Number of records")
    ax.set_ylabel(column.replace("_", " ").title())
    return ax


def plot_numeric(data, column, *, title, bins=30, log_x=False, ax=None):
    """Histogram for a numeric variable; choose log_x only when the scale is meaningful."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    values = data[column].dropna()
    ax.hist(values, bins=bins, color="#4B8A78", edgecolor="white")
    if log_x:
        ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_ylabel("Number of records")
    return ax

## 3.8 Univariate analysis — what does each field look like?

**Source table:** `master` for event-level fields. Use `gaza_weekly` for the weekly distribution and `incidents` only for spatial-input quality. Begin with the distribution of each variable before asking what differs between groups.

Report counts/shares for categories and robust summaries for skewed counts. Show fatalities both as recorded and in the `log1p` display. Do not interpret precision codes as continuous measurements.

In [ ]:
# Profile every field in the cleaned event master, then summarize key descriptive variables.
profile = pd.DataFrame({
    "dtype": master.dtypes.astype(str),
    "non_missing": master.notna().sum(),
    "missing": master.isna().sum(),
    "missing_pct": (master.isna().mean() * 100).round(2),
    "distinct_including_missing": master.nunique(dropna=False),
}).sort_index()
display(profile)

# Fatalities are a skewed count: report robust percentiles and retain the recorded maximum.
fatality_summary = master.fatalities.describe(percentiles=[.5, .75, .9, .95, .99]).to_frame("fatalities")
fatality_summary.loc["zero_count"] = master.fatalities.eq(0).sum()
fatality_summary.loc["positive_count"] = master.fatalities.gt(0).sum()
display(fatality_summary)

# Frequency tables report counts and percentages of all event records.
# High-cardinality actor fields show their 15 most frequent labels for readability.
categorical_fields = ["event_type", "sub_event_type", "disorder_type",
                      "actor1", "actor2", "civilian_targeting",
                      "geo_precision", "time_precision", "admin2", "period"]
for col in categorical_fields:
    counts = master[col].value_counts(dropna=False)
    table = pd.DataFrame({"count": counts, "share_pct": (counts / len(master) * 100).round(2)})
    print(f"\n{col}: {len(counts)} levels (showing up to 15)")
    display(table.head(15))


### Fatality distributions

The first plot shows fatalities on the original count scale, including zeros and the full observed range. The second uses `log1p(fatalities)` so zero values remain visible while large counts are compressed. Both plots use the same event records.


In [ ]:
# V1: event-level fatality distribution, shown on raw and log1p scales.
plot_numeric(master, "fatalities", title="How are reported fatalities distributed across Gaza event records?", bins=50)
plt.tight_layout()
plt.show()

plot_numeric(master, "fatalities_log1p", title="How does the fatality distribution look on a log1p scale?", bins=40)
plt.xlabel("log1p(reported fatalities)")
plt.tight_layout()
plt.show()

# V2: event-type counts and shares. Categories are labels, not an ordered numeric scale.
event_counts = master.event_type.value_counts().sort_values()
event_shares = event_counts / len(master) * 100
ax = event_counts.plot(kind="barh", figsize=(10, 5), color="#315A7D")
ax.set_title("Which event types account for the most Gaza event records?")
ax.set_xlabel("Number of event records")
ax.set_ylabel("ACLED event type")
ax.set_xlim(0, event_counts.max() * 1.4)
for y, (count, share) in enumerate(zip(event_counts, event_shares.loc[event_counts.index])):
    ax.text(count, y, f"  {count:,} ({share:.1f}%)", va="center", fontsize=9)
plt.tight_layout()
plt.show()

# V3: district counts; missing district is kept in the frequency table above.
district_counts = master.admin2.value_counts().sort_values()
ax = district_counts.plot(kind="barh", figsize=(8, 4.5), color="#4B8A78")
ax.set_title("How are Gaza event records distributed across recorded districts?")
ax.set_xlabel("Number of event records")
ax.set_ylabel("Recorded district (admin2)")
plt.tight_layout()
plt.show()

# V4: precision codes are categories; do not interpret them as a numeric scale.
precision_counts = master.geo_precision.value_counts().sort_index()
ax = precision_counts.plot(kind="bar", figsize=(6, 4), color="#D58A32")
ax.set_title("How many event records have each geographic precision code?")
ax.set_xlabel("ACLED geographic precision code")
ax.set_ylabel("Number of event records")
ax.set_xticks(range(len(precision_counts)))
ax.set_xticklabels([str(x) for x in precision_counts.index], rotation=0)
plt.tight_layout()
plt.show()


### Univariate interpretation

**Fatalities (26,664 records):** The median is 0, the 75th percentile is 2, the 95th percentile is 12, and the maximum is 471. A total of 17,634 records (66.1%) report zero fatalities. The distribution is strongly right-skewed: most records have zero or low reported counts, while a small number extend the upper tail. The log1p view makes the lower range easier to inspect while preserving zero. High values remain included; the Phase 2 review flag is not a reason to delete records.

**Event composition:** Explosions/Remote violence is the largest event type (20,276; 76.1%), followed by Strategic developments (3,517; 13.2%) and Battles (1,648; 6.2%). These percentages describe this downloaded event table, not all real-world incidents.

**Place and precision:** Gaza City (8,036), Khan Yunis (6,005), Deir El Balah (4,781), North Gaza (4,303), and Rafah (3,535) account for the recorded districts; four records have no district. Geographic precision is code 1 for 14,313 records (53.7%), code 2 for 12,231 (45.9%), and code 3 for 120 (0.5%). Precision codes are categories with different location certainty, not a measurement scale.

**Why it matters:** These distributions establish the event-level baseline for project questions about recorded event composition, severity, and location. They support robust summaries and showing counts with percentages in later comparisons. ACLED is an event-recording source; counts may reflect reporting and coding coverage. This descriptive analysis does not establish intent, responsibility, or causation.


## 3.9 Bivariate analysis — how do groups compare?

**Source table:** `master` unless the comparison is explicitly weekly or spatial. Start with Phase 1 questions: before/after period differences, event-type composition, actor/event profiles, severity, and precision. Use counts and rates together when group sizes differ.

For every share, state the denominator. For actor comparisons, describe recorded labels and consult notes before assigning actor roles. These are exploratory comparisons; an association is not a causal effect.

In [ ]:
# Example: event-type denominators and focal-event counts/shares.
event_type_summary = master.groupby("event_type", dropna=False).agg(
    events=("event_id_cnty", "size"),
    political_events=("political_violence", "sum"),
    focal_events=("focal_event", "sum"),
)
event_type_summary["focal_share_of_all_events"] = event_type_summary.focal_events / event_type_summary.events
event_type_summary["focal_share_of_political_events"] = (
    event_type_summary.focal_events / event_type_summary.political_events.replace(0, np.nan)
)
display(event_type_summary.sort_values("focal_events", ascending=False))

ax = event_type_summary.sort_values("focal_share_of_political_events")["focal_share_of_political_events"].plot(
    kind="barh", figsize=(9, 5), color="#315A7D"
)
ax.set_title("Within each event type, what share of political-violence records are focal events?")
ax.set_xlabel("Focal events / political-violence events")
ax.set_ylabel("Event type")
plt.tight_layout()
plt.show()

# Interpretation: [Which event types have the largest focal shares? How do counts and denominators affect the comparison?]

### Interpretation — event type and focal-event share

**Observed pattern:** [Identify event types with high counts and/or high within-type focal shares. Quote their denominators.]  
**Why it matters:** [Explain how this helps answer BQ2 and what should be reviewed in event notes.]  
**Limit:** [State how category size, coding, or missingness affects the comparison.]

## 3.10 Grouped, temporal, and spatial analysis — where and when do patterns occur?

This section changes the unit of analysis. `gaza_weekly` is one Gaza-wide week; `district_weekly` is one known district-week; `incidents` is one post–October 7 land focal incident. Keep each figure tied to its table and unit.

For temporal figures, distinguish event counts from shares and mark the incomplete final week. For district comparisons, show denominators and keep unknown-district records in Gaza-wide totals. For spatial figures, compare precision subsets on the same map extent and describe recorded concentrations rather than exact attack points.

In [ ]:
# Weekly and spatial examples; count, share, and coordinate plots have different units.
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(gaza_weekly.week_start, gaza_weekly.focal_events, color="#8D3B42")
axes[0].set_title("How does the number of recorded focal events change by Gaza week?")
axes[0].set_ylabel("Focal events")
axes[1].plot(gaza_weekly.week_start, gaza_weekly.focal_share, color="#315A7D")
axes[1].set_title("What share of recorded political-violence events are focal each week?")
axes[1].set_ylabel("Focal share")
axes[1].set_xlabel("Week start")
plt.tight_layout()
plt.show()

# District-week heatmap: select complete weeks and order districts explicitly.
heat = district_weekly.pivot(index="admin2", columns="week_start", values="focal_events")
plt.figure(figsize=(15, max(4, .45 * len(heat))))
sns.heatmap(heat, cmap="mako", cbar_kws={"label": "Recorded focal events"})
plt.title("Which Gaza districts have high focal-event weeks, and when?")
plt.xlabel("Week start")
plt.ylabel("District (admin2)")
plt.tight_layout()
plt.show()

# Spatial scatter; map projection/basemap may be added if available and sourced.
plt.figure(figsize=(7, 8))
plt.scatter(incidents.longitude, incidents.latitude, s=8, alpha=.35, color="#8D3B42")
plt.title("Where are post–October 7 land focal incidents recorded?")
plt.xlabel("Longitude (degrees)")
plt.ylabel("Latitude (degrees)")
plt.tight_layout()
plt.show()

### Interpretation prompts for grouped, temporal, and spatial figures

**Weekly trend:** [Describe sustained changes, spikes, zero-denominator weeks, and incomplete coverage.]  
**District-week heatmap:** [Identify visible concentration or persistence; distinguish high volume from high share.]  
**Spatial view:** [Describe coordinate concentration and precision limits; do not infer exact boundaries or safety.]

## 3.11 Hypothesis testing plan and report

Set the hypothesis before looking at the test result. Use **at least two** tests aligned with the business questions. Report the test statistic, p-value, effect size, assumptions, and a plain-language conclusion. Treat tests as exploratory because events can be related by time and place.

### Hypothesis A — before/after focal share
- **Question:** Did the share of political-violence event records classified as focal differ between the two periods?
- **H0:** Period and focal-event status are independent among political-violence records.
- **H1:** Period and focal-event status are associated.
- **Table:** Rows = period; columns = focal yes/no; restrict denominator to `political_violence == True`.
- **Candidate test:** Chi-square independence if expected counts are adequate; exact/Monte Carlo alternative if not. Report Cramér’s V.
- **Caution:** Periods have unequal duration, and earlier focal counts may be sparse. Show counts and per-week context; do not imply period caused the difference.

### Hypothesis B — event type and focal status
- **Question:** Does focal-event prevalence vary by recorded event type?
- **H0:** Event type and focal-event status are independent among political-violence records.
- **H1:** At least one event type has a different focal-event prevalence.
- **Table:** Rows = event type; columns = focal yes/no; denominator restricted to political-violence records.
- **Candidate test:** Chi-square independence; use Monte Carlo resampling if expected counts are small. Report Cramér’s V and within-type rates.
- **Caution:** The test is an association in this export. It does not establish event-type effects or causal relationships.

### Test report table — fill after running

| Hypothesis | Test and assumptions checked | Statistic | p-value | Effect size | Plain-language conclusion / limitation |
|---|---|---:|---:|---:|---|
| A: period × focal status | [Fill] | [Fill] | [Fill] | [Fill] | [Fill] |
| B: event type × focal status | [Fill] | [Fill] | [Fill] | [Fill] | [Fill] |

Use a preselected significance threshold (for example, α = 0.05) and disclose multiple exploratory comparisons if you run additional tests.

In [ ]:
# Test scaffold: use political-violence records as the denominator; inspect expected counts before choosing the final procedure.
from scipy.stats import chi2_contingency

political = master.loc[master.political_violence].copy()
period_table = pd.crosstab(political.period, political.focal_event)
print("Period × focal status")
display(period_table)
period_chi = chi2_contingency(period_table, correction=False)
print("Chi-square:", period_chi.statistic, "p:", period_chi.pvalue)
print("Expected counts:")
display(pd.DataFrame(period_chi.expected_freq, index=period_table.index, columns=period_table.columns))

# If expected counts are small, do not blindly report this asymptotic p-value;
# use a justified exact/Monte Carlo procedure and document the method and seed.
# Repeat the same diagnostic for event_type × focal_event before interpreting.

## 3.12 Key findings and Phase 4 plan

### Key insights (final submission: 5–8 evidence-backed bullets)
1. **[Insight]** [Metric/figure/table and which business question it answers.]  
2. **[Insight]** [Evidence and limitation.]  
3. **[Insight]** [Evidence and limitation.]  
4. **[Insight]** [Evidence and limitation.]  
5. **[Insight]** [Evidence and limitation.]  
6. **[Optional insight]** [...]  
7. **[Optional insight]** [...]  
8. **[Optional insight]** [...]

### Phase 4 modeling plan

| Proposed technique | EDA evidence that supports it | Inputs / target | Baseline or comparison | Risks / sensitivity checks |
|---|---|---|---|---|
| DBSCAN spatial clustering | [Concentration, repeated locations, precision pattern] | Land focal-event coordinates in radians; Haversine distance | District-level count summaries | Vary epsilon/min_samples; compare precision-1 subset; interpret clusters as recorded concentrations. |
| Weekly anomaly detection | [Visible spikes/persistence and historical distribution] | District-week focal counts plus eligible prior-week history | Trailing median/MAD rule | Chronological evaluation; incomplete weeks and sparse history; compare with Isolation Forest if retained. |

Only retain techniques supported by the EDA and Phase 1 plan. Record any revisions to the modeling questions and why the revision follows from evidence.

## 3.13 Limitations, reproducibility, and references

### Limitations to report
- ACLED is a recorded-event dataset and may underrepresent events; reporting coverage can change across time and place.
- Fatality totals are event-level reported values and do not automatically identify civilian fatalities.
- Actor, targeting, location, and precision fields reflect the source’s coding and may be incomplete or approximate.
- Observations may be related across time and geography, limiting simple independence assumptions in statistical tests.
- Descriptive differences and statistical associations do not establish causation, intent, or legal classification.

### Reproducibility checklist
- [ ] Phase 2 exports are loaded from `data/processed` and source snapshot is identified.
- [ ] Date parsing and precision category restoration are explicit.
- [ ] Every subset/filter is described, including coast exclusion and district missingness.
- [ ] Randomized tests include a fixed seed and number of resamples.
- [ ] Notebook runs from top to bottom and all charts/tables reconcile to their analysis table.

### References
- Group 7, *Phase 1 — Dataset Selection and Business Problem*.
- Group 7, *Phase 2 — Data Preprocessing*.
- *DS312 Final Project Guide*, Phase 3: Exploratory Data Analysis (EDA).
- ACLED, *Codebook* and the documented event-level export snapshot.